In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout

In [2]:
# Load the cleaned data from your previous notebook
df = pd.read_csv('../data/cleaned_train.csv')

# Drop any NaNs that might have appeared during CSV saving
df = df.dropna(subset=['sentiment', 'text', 'aspect'])

# Combine text and aspect for context
df['combined_text'] = df['text'] + " [SEP] " + df['aspect']

# Split into train and test (since your original test file was empty)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])

X_train_raw = train_df['combined_text'].values
X_test_raw = test_df['combined_text'].values

# Encode labels to integers (0, 1, 2)
le = LabelEncoder()
y_train = le.fit_transform(train_df['sentiment'])
y_test = le.transform(test_df['sentiment'])

print(f"Classes: {le.classes_}")

Classes: ['negative' 'neutral' 'positive']


In [3]:
# Configuration
MAX_WORDS = 5000 
MAX_LEN = 60 # Longest sentence + aspect length

tokenizer = Tokenizer(num_words=MAX_WORDS, lower=True, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_raw)

# Convert to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_test_seq = tokenizer.texts_to_sequences(X_test_raw)

# Pad to ensure uniform length
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print("Tokenization complete.")

Tokenization complete.


In [5]:
# Glove
GLOVE_PATH = r"C:\Users\ananya\Downloads\glove embedding\glove.6B.100d.txt" 
EMBEDDING_DIM = 100

# 1. Load GloVe into a dictionary
embeddings_index = {}
with open(GLOVE_PATH, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# 2. Prepare the matrix for our specific vocabulary
word_index = tokenizer.word_index
embedding_matrix = np.zeros((len(word_index) + 1, EMBEDDING_DIM))

for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [10]:
from tensorflow.keras.layers import SpatialDropout1D, Bidirectional, LSTM, Dense, Dropout, Embedding
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential

# 1. Define hyperparameters
# len(word_index) + 1 ensures we account for the padding token
vocab_size = len(tokenizer.word_index) + 1 

# 2. Build the Model
model = Sequential([
    Embedding(input_dim=vocab_size, 
              output_dim=100, 
              weights=[embedding_matrix], 
              input_length=MAX_LEN, 
              trainable=False),
    
    SpatialDropout1D(0.3), 
    
    # Single layer with 64 units - much more stable for small data
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    
    Dense(32, activation='relu'),
    Dropout(0.5),   
    Dense(3, activation='softmax')
])


# 3. Compile
# Using a slightly lower learning rate (0.001) is usually standard for GloVe
model.compile(loss='sparse_categorical_crossentropy', 
              optimizer=Adam(learning_rate=0.001), 
              metrics=['accuracy'])

model.summary()



c:\Users\ananya\Downloads\absa nlp\absa_nlp\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │       357,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 357,400 (1.36 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 357,400 (1.36 MB)

In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(enumerate(weights))
print(class_weight_dict)

{0: np.float64(1.49120082815735), 1: np.float64(1.8978919631093545), 2: np.float64(0.5547852878875409)}


In [12]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

history = model.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_test_pad, y_test),
    callbacks=[early_stop],
)

Epoch 1/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 14s 62ms/step - accuracy: 0.5720 - loss: 0.9772 - val_accuracy: 0.6019 - val_loss: 0.9063
Epoch 2/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - accuracy: 0.6001 - loss: 0.9242 - val_accuracy: 0.6006 - val_loss: 0.8723
Epoch 3/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - accuracy: 0.6022 - loss: 0.9055 - val_accuracy: 0.6019 - val_loss: 0.8532
Epoch 4/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.6071 - loss: 0.8864 - val_accuracy: 0.6103 - val_loss: 0.8235
Epoch 5/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.6085 - loss: 0.8705 - val_accuracy: 0.6186 - val_loss: 0.8213
Epoch 6/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 0.6126 - loss: 0.8624 - val_accuracy: 0.6283 - val_loss: 0.8285
Epoch 7/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.6230 - loss: 0.8424 - val_accuracy: 0.6158 - val_loss: 0.8222
Epoch 8/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 6s 66ms/step - accuracy: 0.6182 - loss: 0.8374 - val_accuracy: 0.6394 - 